# Brain-state transitions vs. anxiety and depression

Cam-CAN HADS (anxiety and depression categories) against the brain-state transitions, from the
general to the particular:

* **A. Is there a signal in the data, before any model?** Every variable of every state set.
  **A1** compares the movie with rest in the Normal group: the task effect is the largest there is
  and sets the scale. **A2** compares anxiety and depression with the rest of the sample: does
  an association survive every analysis choice?
* **B. Do the models predict?** **B1** shows every row of the four rankings (FC and BSTM, rest and
  movie) against each model's Age + Sex baseline. **B2** shows which modelling choices matter
  (window, atlas, model, state method, arm).
* **C. What kind of information do they use?** Of the models that see every variable type
  (the `all` arms), the share of their SHAP that comes from transitions, occupancy, dynamics and
  covariates.
* **D. One variable space** (atlas × window × state definition, chosen below): what differs, with
  no model. **D1** is the movie vs rest volcano, **D2** the anxiety and depression volcanos, and
  **D3** draws both on the state graph.
* **E. One model in that space** (chosen below): **E1** its score, **E2** what its score looks
  like and whether its ranking of variables is stable, **E3** where it looks on the state graph,
  in the same layout as D3.

**One rule for every highlight.** A variable passes when its BH-FDR < `alpha` **and**
|log2 fold change| ≥ `fc_line`. The fold change is log2((mean + c) / (reference mean + c)), with
a pseudocount c = `shrink` × the block's average for probabilities (cells, occupancy) and `shrink`
× the variable's own mean for the dynamics. It keeps a rare cell that moves from 0.0004 to 0.0010
from looking like a big effect.

The tables come from `hads_tables.py` next to this notebook, and the drawings from
`hads_plots.py`. The slow tables are cached in `cache/`, and `python hads_tables.py` builds them
on the cluster ahead of time. This notebook writes nothing to `outputs/`.

In [ ]:
import hads_tables as ht, hads_plots as hp

cfg = ht.Settings(
    # the statistical rule (every volcano, every diagram)
    case_from="Mild",               # this HADS level and above = the phenotype group
    alpha=0.10,                     # BH-FDR
    fc_line=0.25,                   # |log2 fold change| needed too; None = FDR only
    shrink=1.0,                     # log2fc pseudocount: 0 = raw ratio (widest), 5+ = narrow
    min_present=0.10,               # drop cells / occupancies non-zero in < this share of subjects
    covariates=["Age", "Sex", "n_transitions"],
    # A: the multiverses
    multiverse_arm="all [cond]",    # its variables: cells as p(j | i), occupancy, dynamics
    xlim=2,                         # same x range ±xlim in every volcano; None = fit each
    label_max=25,                   # names per volcano panel
    # B: performance
    models=None,                    # None = every model but logistic
    arms=None,                      # None = every arm, e.g. ["cells [joint]", "edges"]
    baseline_covariates=["Age", "Sex"],
    seeds=[0, 1, 2, 3, 4],          # as select-bstm / select-fcm
    margin_sd=0,                    # "above" = mean - margin_sd * row sd > baseline + its sd
    highlight_max=10,               # rows named per B1 panel; the table has all
    factors={"window": "window size", "atlas": "atlas", "model": "model", "method": "state method", "arm": "arm"},
    factor_trees=["bstm"],          # add "fcm" to pool its rows where the axis exists
    # C: SHAP shares
    share_arms=["all [joint]", "all [cond]"],
    share_top=10,                   # best state sets per (model, arm); None = all (hours of CPU)
    shap_seed=0,
    # D / E
    graph_arms=["all [joint]", "all [cond]"],   # the arms the default choices are drawn from
    edge_top=12,                    # the Normal group's strongest arrows per task
    label_pp=None,                  # None = every highlight shows its pp; e.g. 12 = the largest only
    top_vars=15,                    # E2.2: variables per panel
    seeds_stab=[0, 1, 2, 3, 4],     # E2.2 / E3: fold seeds
    highlight_top=8,                # E3: the model's top graph variables by mean |SHAP|
    stable_min=0.8,                 # E3: ... in the top for this share of the seeds
    min_d=0.2,                      # E3: ... and |Cohen's d| vs Normal at least this
    save_dir=None,                  # e.g. "figures/10" to save PNG + SVG
)
st = ht.Study(cfg)
T0, T1 = cfg.tasks                  # the base task (rest) and the other (movie)
print(f"phenotype: {len(st.pheno):,} subjects;",
      ", ".join(f"{lab} {st.pheno[lab].value_counts().sort_index().astype(int).to_dict()}" for lab in cfg.targets))
# Check this before reading a gap as a result: the runs are separate, so `n` can differ.
st.summ.pivot_table(index=["tree", "task"], columns="phenotype", values="n", aggfunc="max")

## A. The volcanos: every variable of every state set

One dot is one variable in one state set, e.g. `entropy_rate_bits · yeo7 · 30s · HMM1_pca3_8`,
with no ranking or selection. The point is consistency. A dynamics variable whose dots stay on
one side of zero, whatever the atlas, window or method, does not depend on those choices. A hit
in one corner of the grid does. Only the dynamics mean the same thing in every set; state 3 in one
set has nothing to do with state 3 in another, so for cells and occupancy read the cloud, not the dot.

The dashed horizontal line is the FDR cut and the dotted one p = 0.05 uncorrected; the dashed
verticals are ±`fc_line`. Solid dots pass both. FDR runs over every dot of a panel: tens of
thousands of correlated tests, so it is strict (BH stays valid under that positive dependence).

### A1. Movie vs rest, in the Normal group

The people Normal on both HADS scales with both scans, each their own control: movie minus rest
by a paired t-test, so age and sex do not enter.

In A1, A2, D1 and D2 the call takes `alpha=`, `fc_line=` (0 = off) and `xlim=` to change the rule
or the x range for that figure only (nothing is recomputed). The multiverses also take `by=`,
the colouring: `"block"` (cells, occupancy, dynamics), `"window"`, `"atlas"`, `"method"` or `"K"`,
or a list for one figure each.

In [ ]:
mt = st.multiverse_tasks()
hp.fig_multiverse(mt, cfg, f"A1  {T1} vs {T0} · Normal group, paired · every variable of `{cfg.multiverse_arm}` in every set",
                  "A1_multiverse_tasks", f"log2 ({T1} / {T0}), shrink={cfg.shrink:g}",
                  by="block", alpha=None, fc_line=None, xlim=None)    # None = as in cfg
ht.dynamics_consistency(mt)

### A2. Anxiety and depression, the multiverse

The same cloud against each phenotype: `case_from` and above vs everyone below, adjusted for the
covariates by regression, in each task.

In [ ]:
multi = st.multiverse()
hp.fig_multiverse(multi, cfg, f"A2  multiverse · every variable of `{cfg.multiverse_arm}` in every set",
                  "A2_multiverse", f"log2 fold change  ({cfg.case_from}+ vs below, adjusted, shrink={cfg.shrink:g})",
                  by="block", alpha=None, fc_line=None, xlim=None)
ht.dynamics_consistency(multi)

## B. Model performance: every model

### B1. The four rankings

`select-bstm` (transition matrices) and `select-fcm` (static connectivity) at rest and in the
movie, every row of their `summary.csv`. There is one block of panels per model, because every row
already carries the covariates and stands on its own model's baseline. The dashed line is that
model's covariates-only baseline (Age + Sex, fitted here with the pipeline's own CV), with a ± sd
band. The large markers are each tree's best row at rest (●) and in the movie (◆): filled where it
won, hollow where it is carried over. Coloured rows (± sd over seeds) are above the baseline + sd
at rest **and** in the movie; the best `highlight_max` are named, and the table lists all.

In [ ]:
base = st.baselines()
rows = st.performance_rows(base)
hp.fig_rankings(rows, cfg, st.above_both)
st.above_table(rows)

### B2. Which modelling choices matter

One figure per axis of the grid (`factors`), on the same rows as B1 (`factor_trees`). One row of
panels per model, over its baseline; rest and movie side by side in each group. The marker is the
median and the bar the middle half. A group is pooled over everything else in the grid: if one
window only ever ran with a strong atlas, it inherits that atlas's scores, so read a gap as a lead
to check in B1, not as an effect.

In [ ]:
hp.fig_factors(rows, cfg)

## C. Where the models' contribution comes from: the `all` arms

The `all` arms give a model the covariates, the transitions, the occupancy and the dynamics at
once, so the share of its out-of-fold SHAP that each type takes says which one it **chooses** to
lean on. These are the `share_top` best state sets per model and arm, grouped by model type, best
CV score first. The cells block sums many small contributions (K² of them), so compare it with
care against the eight dynamics.

In [ ]:
contrib = st.shares()
hp.fig_shares(contrib, cfg, f"C  the all arms: which variable type the model leans on (top {cfg.share_top} per model and arm)")

## D. One variable space

**Choose it here.** The options below are, for each state set, its best `all`-arm model by
CV score averaged over the four panels, the worst panel's margin over that model's baseline + sd
(`above_all` = above in every panel), and how many of its variables pass in A2. Copy a line into
`SPACE`; `None` takes the first. E then picks a model in this same space.

In [ ]:
options = st.space_options(base, multi)
for _, r in options.iterrows():
    print(f"SPACE = {ht.as_choice(r, ['atlas', 'window_s', 'states'])}   # {r['model']} · {r['arm']}  ρ={r['cv']:.3f}")
SPACE = None                        # e.g. dict(atlas="networks", window_s=30.0, states="HMM1_pca3_8")
space = st.choose_space(SPACE, options)
print("\nspace:", ht.set_label(space))
options

### D1. Movie vs rest

A1's test on this space only: every p(j | i), occupancy and dynamics variable, movie minus rest in
the same Normal people (paired t-test). FDR over the variables of this space. Every dynamics
variable is named, so this is where to read whether entropy, dwell or the switch rate move with
the film.

In [ ]:
paired = st.paired(space)
hp.fig_volcanos(paired, cfg, f"D1  {T1} vs {T0} · Normal group, paired · {ht.set_label(space)}",
                "D1_volcano_tasks", f"log2 ({T1} / {T0}), shrink={cfg.shrink:g}", alpha=None, fc_line=None, xlim=None)
paired.sort_values("p").round(4)

### D2. Anxiety and depression vs Normal

Each phenotype group (`case_from`+ on that scale) against the people Normal on **both** scales,
per task, adjusted for the covariates: the same groups as the diagrams, so a point outlined here
is a red item in D3. FDR per panel over every variable of the space.

In [ ]:
grp = st.group_tests(space)
hp.fig_volcanos(grp, cfg, f"D2  {cfg.case_from}+ vs Normal (on both scales), adjusted · {ht.set_label(space)}",
                "D2_volcano_groups", f"log2 fold change  ({cfg.case_from}+ vs Normal, adjusted, shrink={cfg.shrink:g})",
                alpha=None, fc_line=None, xlim=None)
grp.sort_values("p").groupby(["task", "phenotype"]).head(5).round(4)

### D3. The state graph

Columns are the tasks, rows the groups; each diagram is that group's own dynamics, pooled over
its subjects. **Circle area** is the time spent in a state (the % beside it), **the loop** is
p(stay), thicker when the state is held longer, and **arrow width** is p(j | i), the share of
moves out of i that go to j. Every diagram draws the Normal group's `edge_top` strongest arrows of
each task plus its own highlighted ones, on one width scale for the whole figure.

* **Blue**: the movie's Normal diagram against rest's, D1's passing items. With hundreds of
  people, each their own control, far more passes here than between groups, so `TASK_RULE` makes
  the blue stricter: `alpha` (FDR), `fc_line` (|log2FC|; 1 = a twofold change) and `top` (only the
  N largest |log2FC| that pass). `None` keeps the usual rule. The red never changes with it.
* **Red**: each phenotype against the Normal diagram of its own task, D2's passing items.
* The label is the change in percentage points: beside a circle (`45% (-16 pp)`), at a loop's
  corner (`-20 pp`, the change in p(stay)) or beside an arrow (`+21 pp`, the change in p(j | i)).
  A highlight that rounds to 0 pp is drawn without a number. Time in a state and p(stay) move
  together but are different numbers: leaving a state more often lowers p(stay) first.
* The test of the blue uses the people with both scans; each diagram shows everyone with that scan.
* A highlighted arrow keeps its width: the colour marks it, the width is still p(j | i).

In [ ]:
pooled = st.pooled(space)
TASK_RULE = dict(alpha=None, fc_line=None, top=None)   # the blue only; e.g. fc_line=1 (twofold), top=10
marks = st.stat_marks(pooled, paired, grp, task_alpha=TASK_RULE["alpha"], task_fc=TASK_RULE["fc_line"],
                      task_top=TASK_RULE["top"])
refs = {(T1, "Normal"): (T0, "Normal"), **{(t, lab): (t, "Normal") for t in cfg.tasks for lab in cfg.targets}}
colours = {k: hp.BLUE if k[1] == "Normal" else hp.RED for k in refs}
blue_rule = (f"FDR<{TASK_RULE['alpha'] or cfg.alpha:g} & |log2FC|≥{cfg.fc_line if TASK_RULE['fc_line'] is None else TASK_RULE['fc_line']:g}"
             + (f", the {TASK_RULE['top']} largest" if TASK_RULE["top"] else ""))
hp.fig_graph(pooled, marks, cfg, refs, colours, f"D3  what differs · {ht.set_label(space)}",
             [(hp.BLUE, f"blue = {T1} differs from {T0} (D1: paired, {blue_rule}); label = change in pp"),
              (hp.RED, f"red = differs from Normal (D2: adjusted, {cfg.pass_text}); label = change in pp")],
             "D3_graph")
marks.round(4)

## E. One model in that space

**Choose it here.** Every (model, arm) scored on the space in all four panels, best mean CV first;
only arms with transitions or occupancy, so E3 has something to draw. Copy a line into `MODEL`;
`None` takes the best of `graph_arms`.

In [ ]:
model_options = st.model_options(space)
for _, r in model_options.iterrows():
    print(f"MODEL = {ht.as_choice(r, ['model', 'arm'])}   # ρ={r['cv']:.3f}")
MODEL = None                        # e.g. dict(model="hgb", arm="all [cond]")
model = st.choose_model(MODEL, model_options)
label = f"{model['model']} · {model['arm']} · {ht.set_label(space)}"
print("\nmodel:", label)
model_options

### E1. How it scores: anxiety vs depression, rest vs movie, bstm vs fcm

One model (`E1_MODEL`, hgb by default) after the model selection: the **bstm** row is this space
with the chosen arm, and the **fcm** row is static connectivity on the same atlas with the same
model and arm `FC_ARM` (`edges`, every pairwise correlation; `global`, each person's mean and SD
over those correlations, is its control). One panel per phenotype, a line per task from bstm to
fcm, mean ± sd over the fold seeds, over that model's Age + Sex baseline. When fcm has no run on
this atlas, only bstm is drawn.

In [ ]:
E1_MODEL = "hgb"                    # or "ridge"
FC_ARM = "edges"                    # or "global" (each person's mean and SD over the edges)
perf = st.performance_trees(space, model["arm"], E1_MODEL, base, fc_arm=FC_ARM)
fc_note = "" if (perf["tree"] == "fcm").any() else f" -- no fcm run on {space['atlas']}"
hp.fig_performance(perf, cfg, f"E1  {E1_MODEL}: anxiety vs depression · bstm → fcm · rest vs movie",
                   {"bstm": f"brain-state transitions, selected: {ht.set_label(space)} · {model['arm']}",
                    "fcm": f"static connectivity, same atlas ({space['atlas']}) and model · {FC_ARM}{fc_note}"},
                   E1_MODEL)

### E2. What the score means, and whether its ranking is stable

The model is refit in every panel with `select-bstm`'s own folds, and explained with out-of-fold
SHAP (each fold's model only explains subjects it did not see), in units of the predicted HADS
level: exact for ridge (coefficient × standardised value) and for trees (`shap.TreeExplainer`).
The table checks that the refit **is** the ranked model: its out-of-fold ρ must equal the one
`select-bstm` recorded in `scores.csv`.

**E2.1** is the out-of-fold prediction against the observed category: a bar for the middle half,
a marker for the median. If the categories shift only a little, the model ranks people slightly
better than chance but cannot tell an individual's category.

**E2.2** repeats the SHAP over `seeds_stab`, which change only how subjects are split into folds.
For each panel's `top_vars` variables, one dot per seed and a bar for the average. On the right,
the direction (↑ higher values push the predicted symptoms up, ↓ down, ↕ the seeds disagree) and
how many seeds put it in their own top `top_vars`. A variable there for one seed only should not
be interpreted.

In [ ]:
deep = st.deep(space, model)
st.check(deep)

In [ ]:
hp.fig_predicted(deep, cfg, f"E2.1  observed category vs out-of-fold prediction · {label}")
hp.fig_stability(deep, cfg, f"E2.2  stability over fold seeds · {label}")

### E3. Where the model looks

D3's layout, D3's diagrams. **Red is what the model uses**, on the phenotype rows only (the model
says nothing about the Normal group): the model's `highlight_top` transitions, stays and
occupancies by mean |SHAP|, when the choice is also stable (in the top for at least `stable_min`
of the seeds, as in E2.2) and the group differs from Normal by |Cohen's d| ≥ `min_d` and at least
1 pp. A top variable left grey is one the model relies on although the group means barely differ:
it uses the variation between people, not a group shift. The table lists every candidate and why.

An arrow red in both D3 and E3 is a transition that differs between groups **and** that the
model relies on. For a `[joint]` arm the model's cell is p_joint(i→j) while the arrow shows
p(j | i): the same transition, expressed two ways.

In [ ]:
mmarks = st.model_marks(space, deep, pooled)
refs_model = {(t, lab): (t, "Normal") for t in cfg.tasks for lab in cfg.targets}
hp.fig_graph(pooled, mmarks, cfg, refs_model, {k: hp.RED for k in refs_model}, f"E3  where the model looks · {label}",
             [(hp.RED, f"red = the model's top {cfg.highlight_top} graph variables by mean |SHAP|, top in "
                       f"≥{cfg.stable_min:.0%} of seeds, |Cohen's d| ≥ {cfg.min_d} vs Normal; label = change in pp")],
             "E3_graph")
mmarks.round(3)